# Driving the renderer over its REST API

`peaknav.headless.PeakNavHeadless` starts the real PeakNav renderer off-screen and talks
to it over HTTP. Everything below is one documented request — the client is convenience,
not a protocol of its own; `curl` could do the same.

**This notebook needs more than the elevation one:** a Java runtime, a display (the
window is hidden, but GL still needs a display connection), and the renderer jar. The jar
is fetched on demand into a cache, or taken from a checkout you have built — see
`peaknav.headless.jar` for the search order.

No widgets here; that is notebook 03.

In [ ]:
from peaknav.headless import PeakNavHeadless

ZERMATT = (46.0207, 7.7491)

nav = PeakNavHeadless(*ZERMATT, width=1200, height=700)
nav.status()

The server describes itself. Anything in this notebook that is not in that document is a
bug in one of the two.

In [ ]:
sorted(nav.openapi()["paths"])

## Point the camera and take a picture

`move_to` puts the viewpoint somewhere and, given `await_tiles_ms`, waits for the terrain
to arrive before returning — a frame taken too early is a frame of half-loaded ground.

In [ ]:
nav.move_to(*ZERMATT, await_tiles_ms=120_000)
nav.look(bearing_deg=230, pitch_deg=-4)      # face the Matterhorn
nav.set_altitude_asl(3200)
nav.set_view(sky=True, sky_mode="day", labels=["peaks", "place_names"])
nav.wait(tiles_timeout_ms=60_000, settle_ms=500)

`peaknav.jupyter.show` displays a single frame inline. It needs IPython only — no
ipywidgets — so it works in any notebook.

In [ ]:
from peaknav.jupyter import show

show(nav)

Or write it to a file, choosing the format from the suffix:

In [ ]:
nav.save_frame("matterhorn.jpg")

## The view options

Everything the app can show is a flag on `/view`. Night sky over the same terrain:

In [ ]:
nav.set_view(sky_mode="night", constellations=True, star_names=True,
             sky_time="2026-07-15T23:30:00Z")
nav.wait(settle_ms=500)
show(nav)

## The imagery underneath

The terrain is draped with tiles from an imagery source, and the renderer will take any
XYZ tile server - so the map underneath can be satellite imagery, or an ordinary map. The
sources the app already knows are listed by `GET /providers`:

In [ ]:
[p["id"] for p in nav._request("GET", "/providers")["providers"]]

Anything else is named by its URL template. Here is OpenStreetMap's own cartography,
draped over the same mountains - contours and glaciers and footpaths, in 3D:

In [ ]:
nav.set_view(satellite_template="https://tile.openstreetmap.org/{z}/{x}/{y}.png",
             satellite_name="OpenStreetMap",
             satellite_attribution="© OpenStreetMap contributors")
nav.wait(tiles_timeout_ms=120_000, settle_ms=3000)   # every tile is re-fetched
show(nav)

The template takes `{x}`, `{y}` and `{z}`, and is validated the same way a template typed
into the app is - a malformed one is refused rather than quietly drawing nothing. Passing
the same template again simply re-selects it, so this is safe to re-run.

Please respect the tile server's usage policy: OpenStreetMap's is for modest, personal use
and expects attribution, which is why it is passed above.

Back to a satellite source by id:

In [ ]:
nav.set_view(satellite_provider="LANDSAT")
nav.wait(tiles_timeout_ms=120_000, settle_ms=3000)
show(nav)

## Attaching instead of starting

A renderer started elsewhere — by another notebook, a terminal, a colleague's script — is
attached to by URL. Nothing else in this notebook changes; the client is the same either
way, and `close()` deliberately leaves an attached server running.

```python
nav = PeakNavHeadless.attach("http://127.0.0.1:41123")
```

The port is printed by the renderer when it starts:
`peaknav-headless --lat 46 --lon 7.7 --serve 0` → `PEAKNAV_SERVE port=41123`.

## Shutting down

A spawned renderer is tied to the client's lifetime, so a crashed notebook cannot leave
one running — but closing it explicitly is tidier, and a `with` block does it for you.

In [ ]:
nav.close()

```python
with PeakNavHeadless(*ZERMATT) as nav:
    nav.look(bearing_deg=230, pitch_deg=-4)
    nav.save_frame("matterhorn.png")
```